In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/cleansed")

FLIGHTS_PATH = DATA_DIR / "all_flights_2018-2022.parquet"
WEATHER_PATH = DATA_DIR / "weather_airports_2018_2022_CLEAN.parquet"

flights = pd.read_parquet(FLIGHTS_PATH)
weather = pd.read_parquet(WEATHER_PATH)

In [2]:
flight_cols = ["FlightDate","CRSDepTime","Distance","Cancelled","DepDelayMinutes","Origin"]

flights = pd.read_parquet(FLIGHTS_PATH, columns=flight_cols)
print("flights:", flights.shape)

flights: (29193782, 6)


In [3]:
flights["FlightDate"] = pd.to_datetime(flights["FlightDate"], errors="coerce")
flights = flights.dropna(subset=["FlightDate"])
flights = flights[flights["FlightDate"].dt.year != 2020].copy()

# baseline: excluding cancelled flights for now
flights = flights[flights["Cancelled"] == False].copy()

# label (NO leakage features used in X)
flights["y"] = (flights["DepDelayMinutes"] >= 15).astype(int)

# dep hour
flights["CRSDepTime"] = pd.to_numeric(flights["CRSDepTime"], errors="coerce")
flights["dep_hour"] = (flights["CRSDepTime"] // 100).astype("Int64")
flights["month"] = flights["FlightDate"].dt.month.astype("Int64")
flights["day_of_week"] = flights["FlightDate"].dt.dayofweek.astype("Int64")
flights["date"] = flights["FlightDate"].dt.date

# keeping valid hours + required fields
flights = flights.dropna(subset=["dep_hour","Distance","Origin"])
flights = flights[(flights["dep_hour"] >= 0) & (flights["dep_hour"] <= 23)].copy()

print("after prep:", flights.shape)
print("delay rate:", flights["y"].mean())

after prep: (23695173, 11)
delay rate: 0.18869902321455936


In [4]:
weather = pd.read_parquet(WEATHER_PATH)
print("weather:", weather.shape)

weather["valid"] = pd.to_datetime(weather["valid"], errors="coerce")
weather = weather.dropna(subset=["valid"])

weather["date"] = weather["valid"].dt.date
weather["hour"] = weather["valid"].dt.hour

weather_hourly = (
    weather.groupby(["airport_code","date","hour"])
    .agg({
        "tmpf":"mean",
        "vsby":"mean",
        "sknt":"mean",
        "p01i":"sum",
        "relh":"mean",
        "gust":"max",
    })
    .reset_index()
)

flights = flights.merge(
    weather_hourly,
    left_on=["Origin","date","dep_hour"],
    right_on=["airport_code","date","hour"],
    how="left",
)

flights = flights.drop(columns=["airport_code","hour"], errors="ignore")
print("merged:", flights.shape)
print("weather coverage %:", (1 - flights["tmpf"].isna().mean()) * 100)

weather: (15620936, 22)
merged: (23695173, 17)
weather coverage %: 60.19855605190138


In [5]:
FEATURES_NUM = ["Distance","dep_hour","day_of_week","month","tmpf","vsby","sknt","p01i","relh","gust"]
FEATURES_CAT = ["Origin"]

FEATURES_NUM = [c for c in FEATURES_NUM if c in flights.columns]
FEATURES_CAT = [c for c in FEATURES_CAT if c in flights.columns]

X = flights[FEATURES_NUM + FEATURES_CAT].copy()
y = flights["y"].astype(int).copy()

cutoff = pd.Timestamp("2022-01-01")
train_mask = flights["FlightDate"] < cutoff
test_mask  = flights["FlightDate"] >= cutoff

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test,  y_test  = X.loc[test_mask],  y.loc[test_mask]

print("train:", X_train.shape, "test:", X_test.shape)
print("train y rate:", y_train.mean(), "test y rate:", y_test.mean())

train: (19740047, 11) test: (3955126, 11)
train y rate: 0.1827826955021941 test y rate: 0.21822743447364257


In [6]:
train_n = min(400_000, len(X_train))
test_n  = min(250_000, len(X_test))

X_train_s = X_train.sample(train_n, random_state=42)
y_train_s = y_train.loc[X_train_s.index]

X_test_s = X_test.sample(test_n, random_state=42)
y_test_s = y_test.loc[X_test_s.index]

print(X_train_s.shape, X_test_s.shape)

(400000, 11) (250000, 11)


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

num_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
])

prep = ColumnTransformer(
    [("num", num_tf, FEATURES_NUM), ("cat", cat_tf, FEATURES_CAT)],
    remainder="drop",
    sparse_threshold=0.3
)

model = Pipeline([
    ("prep", prep),
    ("lr", LogisticRegression(
        max_iter=200,
        solver="saga",
        n_jobs=-1,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train_s, y_train_s)
print("fit done ✅")

fit done ✅


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [8]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pred = model.predict(X_test_s)
proba = model.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test_s, pred, digits=3))
print("confusion:\n", confusion_matrix(y_test_s, pred))
print("ROC AUC:", roc_auc_score(y_test_s, proba))

              precision    recall  f1-score   support

           0      0.845     0.641     0.729    195503
           1      0.310     0.577     0.403     54497

    accuracy                          0.627    250000
   macro avg      0.577     0.609     0.566    250000
weighted avg      0.728     0.627     0.658    250000

confusion:
 [[125395  70108]
 [ 23036  31461]]
ROC AUC: 0.6486579328133932


### Baseline Model: Logistic Regression

We trained a logistic regression baseline using flight schedule features
and aggregated hourly weather data. A time-based split (pre-2022 vs 2022)
was used to avoid temporal leakage.

Results:
- ROC AUC ≈ 0.65
- Recall (Delayed flights) ≈ 0.58
- Precision (Delayed flights) ≈ 0.31

This baseline establishes a performance floor and confirms that
both schedule and weather features carry predictive signal.

In [10]:
feature_names = (
    model.named_steps["prep"]
    .get_feature_names_out()
)

coefs = model.named_steps["lr"].coef_[0]

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .sort_values("coef", ascending=False)
)

coef_df.head(10), coef_df.tail(10)

(             feature      coef
 44   cat__Origin_BQN  0.696687
 23   cat__Origin_ASE  0.561315
 73   cat__Origin_DAL  0.537702
 174  cat__Origin_MDW  0.502196
 123  cat__Origin_HOU  0.484189
 1      num__dep_hour  0.476431
 150  cat__Origin_LAS  0.435912
 189  cat__Origin_MQT  0.395964
 256  cat__Origin_SJU  0.337016
 92   cat__Origin_EYW  0.329224,
              feature      coef
 180  cat__Origin_MHT -0.365894
 209  cat__Origin_PDX -0.371405
 97   cat__Origin_FCA -0.399569
 82   cat__Origin_DTW -0.419742
 22   cat__Origin_ANC -0.445492
 199  cat__Origin_OGG -0.720442
 138  cat__Origin_ITO -0.850355
 161  cat__Origin_LIH -0.852139
 147  cat__Origin_KOA -0.891643
 122  cat__Origin_HNL -0.964123)